In [1]:
import scanpy as sc
import scprep
import phate
import numpy as np
import SPARC

import sys
sys.path.append('..')
from run.run_gspa import calculate_wavelet_dictionary
from run.run_ae_default_config import run_ae
from utils import *

### Preprocess

In [32]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20]
print(f"#cells after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

/home/aarthivenkat/.local/lib/python3.8/site-packages/anndata/_core/anndata.py:1830: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


#cells after MT filter: 3861


/home/aarthivenkat/.local/lib/python3.8/site-packages/scanpy/preprocessing/_simple.py:251: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['n_cells'] = number


In [33]:
adata.write('data/V1_Human_Lymph_Node/processed.h5ad')

### Get SPARC operator

In [2]:
adata = sc.read_h5ad('data/V1_Human_Lymph_Node/processed.h5ad')

In [3]:
sparc_op = SPARC.spARC(n_jobs=-1, random_state=42)
data_sparc = sparc_op.fit_transform(expression_X = adata.to_df(),
                                    spatial_X = adata.obs[['array_row', 'array_col']])

Calculating spARC...
  Calculating PCA...
  Calculated PCA in 2.12 seconds.
  Calculating expression graph...
  Calculated expression graph in 0.51 seconds.
  Calculating spatial graph...
  Calculated spatial graph in 0.97 seconds.
  Calculating random walks on expression graph...
  Calculated random walks on expression graph in 0.79 seconds.
  Calculating random walks on spatial graph...
  Calculating spARCed expression data...
  Calculated spARCed expression data in 2.42 seconds.
Calculated spARC in 6.82 seconds.


In [4]:
integrated_diff_op = sparc_op.expression_diff_op @ sparc_op.spatial_diff_op

## GSPA

In [5]:
cell_dictionary, wavelet_sizes = calculate_wavelet_dictionary(P=integrated_diff_op, use_reduced=True)

Maximum scale: 8


100%|██████████| 6/6 [00:41<00:00,  6.85s/it]


In [6]:
data_hvg, hvgs = scprep.select.highly_variable_genes(adata.to_df(), adata.var_names, percentile=90)
data_hvg = data_hvg / np.linalg.norm(data_hvg, axis=0)

In [7]:
N = integrated_diff_op.shape[0]
uniform_signal = np.ones((1, N))
uniform_signal = uniform_signal / np.linalg.norm(uniform_signal, axis=1).reshape(-1,1)

In [8]:
results = {}

signals_projected = project(data_hvg.T, cell_dictionary)
signals_reduced = svd(signals_projected)

In [9]:
results['signal_embedding'] = run_ae(signals_reduced)
np.savez('results/GSPA_QR.npz', signal_embedding=results['signal_embedding'])

Epoch 1/100
59/59 [==============================] - 1s 8ms/step - loss: 0.0034 - val_loss: 0.0029
Epoch 2/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0026 - val_loss: 0.0024
Epoch 3/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0022 - val_loss: 0.0022
Epoch 4/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0021 - val_loss: 0.0021
Epoch 5/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0020 - val_loss: 0.0020
Epoch 6/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0019 - val_loss: 0.0020
Epoch 7/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0019 - val_loss: 0.0019
Epoch 8/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0018 - val_loss: 0.0019
Epoch 9/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0018 - val_loss: 0.0019
Epoch 10/100
59/59 [==============================] - 0s 6ms/step - loss: 0.0018 - val_loss: 0.0019
Epoch 11/

In [10]:
uniform_projected = project(uniform_signal, cell_dictionary)
results['localization_score'] = calculate_localization(uniform_projected, signals_projected)
np.savez('./results/GSPA_QR.npz', signal_embedding=results['signal_embedding'],
         localization_score=results['localization_score'], genes=hvgs)